# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema accessed via a public URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Convert to dict for pretty printing, not for logic
metadata = json.loads(dataset.metadata.to_jsonld())
print(f"Dataset title: {dataset.metadata.name}\n\n{dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Below, we print an overview of all record sets in the dataset, listing each record set `@id` and the fields it contains (by their `@id`).

Use the `@id` to refer to record sets and fields in later steps.

In [ ]:
record_sets = list(dataset.metadata.record_sets)
print(f'Total record sets: {len(record_sets)}')
for rs in record_sets:
    print(f"\nRecord set @id: {rs.id}")
    print(f"  Label     : {rs.label}")
    print(f"  Description: {rs.description}")
    print(f"  Fields:")
    for field in rs.fields:
        print(f"    @id: {field.id} | label: {field.label} | dataType: {field.data_type}")

Let's examine the first few records from the primary record set for illustration. Replace `<record_set_id>` by the appropriate `@id` from the list above.

In [ ]:
# Example: Print the first 2 records from the first record set
if record_sets:
    primary_rs_id = record_sets[0].id
    print(f"Sampling 2 records from record set: {primary_rs_id}\n---")
    for i, record in enumerate(dataset.records(record_set=primary_rs_id)):
        print(f"Record {i+1}: {record}")
        if i >= 1:
            break
else:
    print('No record sets found!')

## 3. Data Extraction
Load data from each record set into a pandas DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# List record set @ids for extraction
rs_ids = [rs.id for rs in record_sets]
dataframes = {}

for rs_id in rs_ids:
    records_iter = dataset.records(record_set=rs_id)
    # Consume generator
    records = list(records_iter)
    if records:
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"Loaded DataFrame for record set: {rs_id} -- {dataframes[rs_id].shape[0]} records, {dataframes[rs_id].shape[1]} columns.")
    else:
        print(f"Record set: {rs_id} is empty.")

# For demonstration, pick the first non-empty dataframe
main_rs_id = None
for rs_id in rs_ids:
    if rs_id in dataframes:
        main_rs_id = rs_id
        break
if main_rs_id:
    print(f"\nColumns in main record set ({main_rs_id}):\n{dataframes[main_rs_id].columns.tolist()}")
    dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
We'll perform data filtering and transformation on a numeric field.

Replace `<numeric_field_id>` and `<group_field_id>` with appropriate `@id`s as listed above. E.g., for a protein mass field, use its column name (usually an `@id`).

In [ ]:
# Example: EDA for primary record set
df = dataframes[main_rs_id]

# Try to find a likely numeric column -- e.g., 'cr:MW' (molecular weight), or 'cr:Peptide count*', etc.
numeric_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col]) or 'count' in col.lower() or 'MW' in col or 'Abundance' in col]
print(f"Numeric field candidates: {numeric_candidates}")
# If available, choose a candidate; else select any numeric column
if numeric_candidates:
    numeric_field = numeric_candidates[0]
else:
    numeric_field = df.select_dtypes('number').columns[0] if len(df.select_dtypes('number').columns) > 0 else df.columns[0]

threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 0
print(f"Filtering {numeric_field} by threshold > {threshold:.2f}")

filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records (>{threshold:.2f}): {filtered_df.shape[0]}")
print(filtered_df[[numeric_field]].head())

# Normalize
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} column:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a categorical field, e.g., if 'cr:Sample' exists
category_candidates = [col for col in df.columns if pd.api.types.is_categorical_dtype(df[col]) or df[col].dtype=='object']
group_field = category_candidates[0] if category_candidates else None

if group_field and group_field != numeric_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
    print(f"\nGrouped data by {group_field} (mean {numeric_field}):")
    print(grouped_df.head())

## 5. Visualization
Visualize the distribution and relationships for the main numeric field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# Histogram of the chosen numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field], kde=True)
plt.xlabel(numeric_field)
plt.title(f"Distribution of {numeric_field}")
plt.show()

# Boxplot by group if possible
if group_field and group_field != numeric_field:
    plt.figure(figsize=(10,4))
    sns.boxplot(x=group_field, y=numeric_field, data=df)
    plt.xticks(rotation=30, ha='right')
    plt.title(f"{numeric_field} by {group_field}")
    plt.show()

## 6. Conclusion

This notebook demonstrated the use of `mlcroissant` to inspect, extract, and analyze records from the FAIR^2 protein mass spectrometry dataset. We reviewed the primary record set fields (by `@id` per Croissant schema), examined their data types, and performed a basic exploratory analysis such as filtering and normalizing a numeric measurement. Visualizations provided an overview of data distributions and possible groupwise variation.

For further work, see [mlcroissant documentation](https://mlcommons.github.io/croissant/api.html) and the dataset's Croissant metadata for detailed schema information. To perform additional analyses, repeat steps above using desired record sets and fields by their `@id`.
